# Hypertension Analytics — Zion Data Center

**Author:** Topaz — Data & Operations Analyst  
**Dataset:** 350 hypertension patients | January – December 2024  
**Objective:** Clinical data analysis covering profiling, cleaning, feature engineering, EDA, statistical testing, insights, and recommendations.

---

## Problem Statements

This analysis is structured around two clinical questions:

### PS1 — BP Control Failure Prediction
**"What predicts BP control failure at 3 months, and can we identify high-risk patients at the point of prescription?"**

- Target variable: `bp_controlled_after_3_months` (1 = controlled, 0 = not controlled)
- Only 9.4% of patients achieved control — making early identification of likely failures critical
- Approach: Feature importance analysis + logistic regression model using variables available at prescription time (Age, Gender, comorbidities, Baseline BP, Drug Class, Medication Adherence)

### PS2 — Drug Class Effectiveness by Comorbidity Profile
**"Which antihypertensive drug class delivers the best outcomes for patients with specific comorbidity profiles?"**

- Outcome: BP control rate and BP reduction by drug class
- Segmentation: Comorbidity profiles (DM only, CKD only, DM+CKD, neither, multi-comorbidity)
- Approach: Cross-tabulation + chi-square test + mean BP reduction by drug class × comorbidity segment

---

*Both questions are answered across Phases 4–7 of this notebook.*

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

# Plotting defaults
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100
sns.set_theme(style='whitegrid', palette='muted')

print("Libraries loaded successfully")

## Phase 1: Data Review & Profiling

In [ ]:
df_raw = pd.read_excel('data/raw/Hypertension_Cardiology_Center_Dataset.xlsx')
df_clean = pd.read_excel('data/raw/Cleaned_data.xlsx')
df_transformed = pd.read_csv('data/raw/Hypertension_Transformed_Data.csv')
print(f"Raw:         {df_raw.shape[0]} rows × {df_raw.shape[1]} cols")
print(f"Cleaned:     {df_clean.shape[0]} rows × {df_clean.shape[1]} cols")
print(f"Transformed: {df_transformed.shape[0]} rows × {df_transformed.shape[1]} cols")

In [ ]:
print("COLUMN INVENTORY")
print(df_raw.dtypes.to_string())
print(f"\nNulls: {df_raw.isnull().sum().sum()}")
print(f"Duplicates: {df_raw.duplicated().sum()}")

In [ ]:
display(df_raw.describe().T.round(2))

In [ ]:
cat_cols = ['Gender','Diabetes_Mellitus','Chronic_Kidney_Disease','Dyslipidemia',
            'Obesity','Antihypertensive_Class','Medication_Adherence','BP_Controlled_After_3_Months']
for col in cat_cols:
    vc = df_raw[col].value_counts()
    pct = df_raw[col].value_counts(normalize=True).mul(100).round(1)
    print(f"\n{col}:")
    print(pd.concat([vc, pct], axis=1, keys=['Count','Pct%']).to_string())

### Data Quality Summary

| Metric | Value |
|---|---|
| Total patients | 350 |
| Variables | 16 |
| Null values | 0 |
| Duplicate rows | 0 |
| Date range | Jan – Dec 2024 |

**Key finding:** Only **9.4% of patients (33/350)** achieved BP control after 3 months — the central clinical finding.

## Phase 2: Data Cleaning

In [ ]:
# Build working copy from raw
df = df_raw.copy()

# Standardise column names: lowercase, underscores
df.columns = (df.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_', regex=False)
    .str.replace(r'[^a-z0-9_]', '', regex=True))

print("Standardised columns:")
print(df.columns.tolist())

In [ ]:
# Cast Registration_Date to datetime (already parsed, but ensure consistency)
df['registration_date'] = pd.to_datetime(df['registration_date'])

# Binary yes/no -> bool (1/0)
binary_cols = ['diabetes_mellitus', 'chronic_kidney_disease', 'dyslipidemia',
               'obesity', 'bp_controlled_after_3_months']
for col in binary_cols:
    df[col] = (df[col].str.strip().str.lower() == 'yes').astype(int)

# Ordered categorical: medication adherence
adherence_order = ['Poor', 'Moderate', 'Good']
df['medication_adherence'] = pd.Categorical(
    df['medication_adherence'], categories=adherence_order, ordered=True)

# Unordered categoricals
for col in ['gender', 'antihypertensive_class']:
    df[col] = pd.Categorical(df[col])

print("Type casting complete")
print(df.dtypes)

In [ ]:
range_checks = {
    'age': (18, 100),
    'baseline_systolic_bp': (100, 220),
    'baseline_diastolic_bp': (60, 140),
    'followup_systolic_bp': (90, 220),
    'followup_diastolic_bp': (60, 140),
    'number_of_visits': (1, 20),
}

print("Range validation:")
all_valid = True
for col, (lo, hi) in range_checks.items():
    out = df[(df[col] < lo) | (df[col] > hi)]
    status = "✓" if len(out) == 0 else f"⚠ {len(out)} outliers"
    print(f"  {col} [{lo}-{hi}]: {status}")
    all_valid = all_valid and len(out) == 0

print(f"
All ranges valid: {all_valid}")

In [ ]:
assert df.isnull().sum().sum() == 0, "Nulls remain!"
assert df.duplicated().sum() == 0, "Duplicates remain!"

print(f"Clean dataset: {df.shape[0]} rows × {df.shape[1]} cols")
print("Zero nulls ✓  Zero duplicates ✓")

# Export
df.to_csv('data/processed/cleaned_data.csv', index=False)
print("Exported: data/processed/cleaned_data.csv")

### Cleaning Summary

| Step | Action | Result |
|---|---|---|
| Column names | Lowercased and snake_cased | 16 columns standardised |
| Registration_Date | Cast to datetime64 | No parse errors |
| Binary columns (5) | Yes/No → 1/0 integer | Clean numeric flags |
| Medication_Adherence | Ordered category (Poor→Moderate→Good) | Ordinal encoding preserved |
| Gender, Antihypertensive_Class | Unordered category | Memory-efficient encoding |
| Range validation | Checked 6 numerical columns | All within clinical bounds |
| Nulls / Duplicates | Post-clean assertion | 0 nulls, 0 duplicates confirmed |

Working dataframe `df` is now clean and ready for feature engineering.

## Phase 3: Feature Engineering & Transformation

In [ ]:
# BP Reduction (treatment effect)
df['sbp_reduction'] = df['baseline_systolic_bp'] - df['followup_systolic_bp']
df['dbp_reduction'] = df['baseline_diastolic_bp'] - df['followup_diastolic_bp']

# Pulse pressure (cardiovascular risk marker)
df['baseline_pulse_pressure'] = df['baseline_systolic_bp'] - df['baseline_diastolic_bp']
df['followup_pulse_pressure'] = df['followup_systolic_bp'] - df['followup_diastolic_bp']

print("BP reduction summary:")
print(df[['sbp_reduction','dbp_reduction']].describe().round(2))

In [ ]:
df['age_band'] = pd.cut(
    df['age'],
    bins=[0, 40, 55, 70, 100],
    labels=['<40', '40-54', '55-69', '70+']
)
print("Age band distribution:")
print(df['age_band'].value_counts().sort_index())

In [ ]:
def comorbidity_profile(row):
    dm = row['diabetes_mellitus']
    ckd = row['chronic_kidney_disease']
    if dm == 1 and ckd == 1:
        return 'DM+CKD'
    elif dm == 1 and ckd == 0:
        return 'DM only'
    elif dm == 0 and ckd == 1:
        return 'CKD only'
    else:
        return 'Neither'

df['comorbidity_profile'] = df.apply(comorbidity_profile, axis=1)
df['comorbidity_count'] = df['diabetes_mellitus'] + df['chronic_kidney_disease'] + df['dyslipidemia'] + df['obesity']

print("Comorbidity profile distribution:")
print(df['comorbidity_profile'].value_counts())
print(f"
Comorbidity count distribution:")
print(df['comorbidity_count'].value_counts().sort_index())

In [ ]:
def classify_baseline_bp(sbp):
    if sbp < 150:
        return 'Mild HTN (140-149)'
    elif sbp < 160:
        return 'Moderate HTN (150-159)'
    elif sbp < 170:
        return 'Moderate-Severe HTN (160-169)'
    else:
        return 'Severe HTN (170+)'

df['baseline_bp_category'] = df['baseline_systolic_bp'].apply(classify_baseline_bp)
print("Baseline BP category:")
print(df['baseline_bp_category'].value_counts())

In [ ]:
# Risk score from variables available AT PRESCRIPTION TIME (no follow-up data)
df['prescription_risk_score'] = (
    (df['age'] > 60).astype(int) +
    df['diabetes_mellitus'] +
    df['chronic_kidney_disease'] +
    df['obesity'] +
    (df['baseline_systolic_bp'] >= 170).astype(int) +
    (df['medication_adherence'] == 'Poor').astype(int)
)

print("Prescription risk score distribution:")
print(df['prescription_risk_score'].value_counts().sort_index())
print(f"
BP failure rate by risk score:")
print(df.groupby('prescription_risk_score')['bp_controlled_after_3_months']
      .apply(lambda x: f'Control rate: {x.mean()*100:.1f}% ({x.sum()}/{len(x)})'))

### Feature Engineering Summary

| Feature | Description | Used in |
|---|---|---|
| `sbp_reduction` | Baseline minus Followup systolic BP | PS1, EDA |
| `dbp_reduction` | Baseline minus Followup diastolic BP | EDA |
| `baseline_pulse_pressure` | Systolic − Diastolic at baseline | EDA |
| `age_band` | 4 age groups: <40, 40-54, 55-69, 70+ | EDA, PS1 |
| `comorbidity_profile` | DM+CKD / DM only / CKD only / Neither | **PS2** |
| `comorbidity_count` | Sum of 4 binary comorbidity flags | PS1, EDA |
| `baseline_bp_category` | 4 severity tiers based on baseline SBP | EDA |
| `prescription_risk_score` | 0-6 composite risk at prescription time | **PS1** |

## Phase 4: Exploratory Data Analysis (EDA)

In [ ]:
# EDA code goes here

## Phase 5: Statistical Analysis

In [ ]:
# Statistical analysis code goes here

## Phase 6: Insights & Interpretation

In [ ]:
# Insights code goes here

## Phase 7: Recommendations

In [ ]:
# Recommendations code goes here